In [2]:
print("Hello")

Hello


In [3]:
!pip install great_tables
import pandas as pd
from great_tables import GT

data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")

/tmp/ipykernel_44283/1432933043.py:5: DtypeWarning: Columns (6,8,19,23,25,29) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")


In [4]:
data.columns
for col in data.columns:
    print(col)

equip_numero
inst_numero
dep_code
reg_code
aps_name


In [5]:
pd.unique(data['equip_type_famille'])

In [6]:
data['equip_type_famille'].str.lower().str.contains("site d'activités aquatiques et nautiques").sum()

In [ ]:

data_nautique = data[data['equip_type_famille'] == "Site d'activités aquatiques et nautiques"]


colonnes = ['equip_numero', "inst_numero", "inst_nom", "inst_adresse", "inst_cp", "dep_code", "reg_code", "dep_nom", "reg_nom", "lib_bdv", "equip_nom", "equip_type_name", "equip_coordonnees", "aps_name"]
data_nautique = data_nautique[colonnes].reset_index(drop=True).copy()



In [ ]:
data_sud = data_nautique[data_nautique['dep_code'].str.lower().isin(["13", "6"])]
data_sud['aps_name'].str.lower().str.contains("surf|ski|voile|foil|wake").sum()

In [ ]:
mots = ["surf", "ski", "voile", "foil", "wake"]
pattern = "|".join(mots)
data_nautique = data_nautique[data_nautique['aps_name'].str.contains(pattern, case=False, na=False)]
data_nautique['dep_code'] = data_nautique['dep_code'].astype(str)
data_nautique

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Séparer lat/lon (si equip_coordonnees est du type "lat, lon")
data_nautique[['lat', 'lon']] = data_nautique['equip_coordonnees'].str.split(',', expand=True).astype(float)

data_nautique['lat'] = data_nautique['lat'].dropna()
data_nautique['lon'] = data_nautique['lon'].dropna()
data_nautique['equip_nom'] = data_nautique['equip_nom'].dropna()
# Créer le GeoDataFrame
geometry = [Point(lon, lat) for lat, lon in zip(data_nautique['lat'], data_nautique['lon'])]
gdf = gpd.GeoDataFrame(data_nautique, geometry=geometry, crs=2154)

In [ ]:
!pip install folium

import folium
m = folium.Map(location=[46.6, 2.3], zoom_start=6)

for _, row in data_nautique.dropna(subset=['lat', 'lon']).iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['aps_name']
    ).add_to(m)

#m


In [18]:
# Nombre d'équipements par département
tab_dep = (
    data_nautique
    .groupby('dep_nom')
    .size()
    .reset_index(name='nb_equipements')
)
tab_dep

In [15]:
# Top 3
top3 = tab_dep.nlargest(3, 'nb_equipements')

# Bottom 3
bottom3 = tab_dep.nsmallest(3, 'nb_equipements')

# Combiner
tab_final = pd.concat([top3, bottom3])

# Ajouter une colonne pour distinguer
tab_final['categorie'] = ['Top 3'] * 3 + ['Les 3 moins bons'] * 3

tab_final = tab_final.sort_values(
    by=['categorie', 'nb_equipements'],
    ascending=[False, False]
)

tab_final = tab_final.reset_index(drop=True)

tab_final

In [ ]:
from great_tables import GT

table = (
    GT(tab_final)
    .tab_header(
        title="Départements les plus et moins équipés d'équipements nautiques)",
        subtitle="Top 3 et Bottom 3 en nombre d'équipements"
    )
    .cols_label(
        dep_nom="Nom du département",
        nb_equipements="Nombre d'équipements",
        categorie="Catégorie"
    )
    .fmt_number(columns=['nb_equipements'], decimals=0)
    .data_color(
        columns=['nb_equipements'],
        palette=["lightblue", "darkblue"]
    )
)

table